**All model runs were completed on A100 GPUs**

**Overview of testing scenario**

Only code change

1.   Only send diff
2.   Send diff and source code file
3.   Send diff, source code file, source test file


Code change and test file change

1.   Only send diff
2.   Send diff and source code file
3.   Send diff, source code file, source test file


Prompt types


1.   Zero shot
2.   Few shot
3.   Instruction
4.   Chain-of-thought


Model sizes


1.   Small
2.   Medium
3.   Large
4.   Extra Large

In [ ]:
# For google models, the latest version of transformers is needed
!pip install -q transformers accelerate bitsandbytes
!pip install --upgrade transformers


# For other models, pinning to 4.48.3 version worked better
#!pip install -q "transformers==4.48.3" "accelerate>=1.0.0" "bitsandbytes>=0.45.0"

In [ ]:
import torch
import difflib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import json
import re
from bs4 import BeautifulSoup
from google.colab import userdata, files
import os
import time
from datetime import datetime
import shutil

In [ ]:
# Note this was added to get access to higher Hugging face rate limits (set the secret in colab with the generated Hugging Face API token)
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

**Few Shot Examples Used in Prompts**

In [ ]:
# Initial few shot examples
FEW_SHOT_EXAMPLES = """
### EXAMPLE 1
GIT DIFF:
+ <button id="cancel-order">Cancel Order</button>
EXISTING TEST FILE:
test('cancel button is functional', async ({ page }) => {
  await page.goto('/');
  await page.locator('#cancel-order').click();
  await expect(page.getByText('Order Cancelled')).toBeVisible();
});

### EXAMPLE 2
GIT DIFF:
- if (count > 0)
+ if (count > 10)
EXISTING TEST FILE:
test('shows warning only when count exceeds 10', async ({ page }) => {
  await page.goto('/');
  await page.fill('#count-input', '5');
  await expect(page.getByText('Warning')).setHidden();
  await page.fill('#count-input', '11');
  await expect(page.getByText('Warning')).toBeVisible();
});
"""

In [ ]:
# Refined few shot examples
# Note inclusion of imports and provision of correct app url (this accounted for most of the improvement)

FEW_SHOT_EXAMPLES_REFINED = """
### EXAMPLE 1
GIT DIFF:
+ <button id="cancel-order">Cancel Order</button>
EXISTING TEST FILE:
import { test, expect } from '@playwright/test';

test('cancel button is functional', async ({ page }) => {
  await page.goto('http://localhost:5175');
  await page.locator('#cancel-order').click();
  await expect(page.getByText('Order Cancelled')).toBeVisible();
});

### EXAMPLE 2
GIT DIFF:
- if (count > 0)
+ if (count > 10)
EXISTING TEST FILE:
import { test, expect } from '@playwright/test';

test('shows warning only when count exceeds 10', async ({ page }) => {
  await page.goto('http://localhost:5175');
  await page.fill('#count-input', '5');
  await expect(page.getByText('Warning')).setHidden();
  await page.fill('#count-input', '11');
  await expect(page.getByText('Warning')).toBeVisible();
});
"""

In [ ]:
def generate_prompt(prompt_type, diff, source_code=None, test_file=None):
    """
    Build prompt based on the scenario (provided params) and prompt type.
    """

    # Diff injected into context data variable
    context_data = f"### GIT DIFF:\n{diff}\n"

    # If source code defined, add it to the context data
    if source_code:
        context_data += f"\n### SOURCE CODE FILE:\n{source_code}\n"

    # If test file data defined, add it to the context data
    if test_file:
        context_data += f"\n### EXISTING TEST FILE:\n{test_file}\n"

    # Define prompts for each defined type
    # zero-shot, few-shot, instruction, chain-of-thought
    strategies = {
        "zero-shot": (
            f"Write a Playwright test in TypeScript. Output ONLY the code block. No explanations. Tests for the following changes:\n{context_data}"
        ),
        "few-shot": (
            f"Examples of our testing style:\n{FEW_SHOT_EXAMPLES}\n\n"
            f"Now, generate a Playwright test for this specific change:\n{context_data}"
        ),
        "instruction": (
            "You are a Senior QA Engineer. Generate a Playwright test using these rules:\n"
            "1. Use getByRole or getByText locators.\n"
            "2. Ensure assertions are included.\n"
            "3. Use TypeScript.\n\n"
            f"Data to process:\n{context_data}"
        ),
        "chain-of-thought": (
            "Analyze the following data step-by-step:\n"
            "1. Identify the UI change from the diff.\n"
            "2. Map the change to the implementation in the source code.\n"
            "3. Outline the test steps.\n"
            "4. Write the final Playwright test code.\n\n"
            f"Data:\n{context_data}"
        )
    }

    # Get the prompt by key, default to zero-shot if no match
    return strategies.get(prompt_type.lower(), strategies["zero-shot"])

In [ ]:
def generate_refined_prompt(prompt_type, diff, source_code=None, test_file=None):
    """
    Build prompt based on the scenario (provided params) and prompt type.
    This just uses the updated prompts that were used for the refined runs (which saw way better performance)
    """

    # Diff injected into context data variable
    context_data = f"### GIT DIFF:\n{diff}\n"

    # If source code defined, add it to the context data
    if source_code:
        context_data += f"\n### SOURCE CODE FILE:\n{source_code}\n"

    # If test file data defined, add it to the context data
    if test_file:
        context_data += f"\n### EXISTING TEST FILE:\n{test_file}\n"


    # Define prompts for each defined type
    # zero-shot, few-shot, instruction, chain-of-thought
    # Note inclusion of import instruction and app url were the two key changes
    strategies = {
        "zero-shot": (
            "Write a Playwright test in TypeScript for the following changes.\n\n"
            "IMPORTANT REQUIREMENTS:\n\n"
            "- Start with: import { test, expect } from '@playwright/test';\n"
            "- Use http://localhost:5175 as the base URL for page.goto()\n"
            "- Output ONLY raw TypeScript code — no markdown fences, no explanations\n"
            "- Ensure the code is syntactically valid TypeScript\n\n"
            f"{context_data}"
        ),
        "few-shot": (
            f"Examples of our testing style:\n{FEW_SHOT_EXAMPLES_REFINED}\n\n"
            "IMPORTANT: Always include the import statement. Use http://localhost:5175 as the base URL. Output ONLY raw TypeScript code — no markdown fences.\n"
            f"Now, generate a Playwright test for this specific change:\n{context_data}"
        ),
        "instruction": (
            "You are a Senior QA Engineer. Generate a Playwright test using these rules:\n"
            "1. Start with: import { test, expect } from '@playwright/test';\n"
            "2. Use http://localhost:5175 as the base URL for page.goto()\n"
            "3. Use getByRole or getByText locators.\n"
            "4. Ensure assertions using expect() are included.\n"
            "5. Use TypeScript.\n"
            "6. Output ONLY raw TypeScript code — no markdown fences (```), no explanations.\n"
            f"Data to process:\n{context_data}"
        ),
        "chain-of-thought": (
            "Analyze the following data step-by-step:\n"
            "1. Identify the UI change from the diff.\n"
            "2. Map the change to the implementation in the source code.\n"
            "3. Outline the test steps.\n"
            "4. Write the final Playwright test code.\n\n"
            "IMPORTANT REQUIREMENTS for the final code:\n"
            "- Start with: import { test, expect } from '@playwright/test';\n"
            "- Use http://localhost:5175 as the base URL for page.goto()\n"
            "- Output ONLY raw TypeScript code — no markdown fences (```), no explanations\n"
            "- Ensure the code is syntactically valid TypeScript\n"
            f"Data:\n{context_data}"
        )
    }

    # Get the prompt by key, default to zero-shot if no match
    return strategies.get(prompt_type.lower(), strategies["zero-shot"])

**Base Files**

In [ ]:
# This is the base code file (assumption file is on main branch) - diffs created off of this
BASE_CODE_FILE = """
import React from "react"
import { JSX, useState, useEffect } from "react"
import { api, Note, CreateNoteInput, UpdateNoteInput } from "../services/api"

export function Home(): JSX.Element {
  const [notes, setNotes] = useState<Note[]>([])
  const [selectedNote, setSelectedNote] = useState<Note | null>(null)
  const [loading, setLoading] = useState(false)
  const [error, setError] = useState<string | null>(null)

  const [formTitle, setFormTitle] = useState("")
  const [formBody, setFormBody] = useState("")
  const [formTags, setFormTags] = useState("")
  const [isEditing, setIsEditing] = useState(false)
  const [savedIds, setSavedIds] = useState<number[]>([])

  useEffect(() => {
    const saved = localStorage.getItem("noteIds")
    if (saved) {
      setSavedIds(JSON.parse(saved) as number[])
    }
  }, [])

  const saveNoteId = (id: number) => {
    const updated = [...savedIds]
    if (!updated.includes(id)) {
      updated.push(id)
      setSavedIds(updated)
      localStorage.setItem("noteIds", JSON.stringify(updated))
    }
  }

  const removeNoteId = (id: number) => {
    const updated = savedIds.filter(nid => nid !== id)
    setSavedIds(updated)
    localStorage.setItem("noteIds", JSON.stringify(updated))
  }

  const loadNote = async (id: number) => {
    setLoading(true)
    setError(null)
    try {
      const note = await api.getNote(id)
      setSelectedNote(note)
      setIsEditing(false)
    } catch (err) {
      setError(err instanceof Error ? err.message : "Failed to load note")
    } finally {
      setLoading(false)
    }
  }

  const handleCreateNote = async (e: React.FormEvent) => {
    e.preventDefault()
    setLoading(true)
    setError(null)

    try {
      const tags = formTags
        .split(",")
        .map(t => t.trim())
        .filter(t => t)
      const input: CreateNoteInput = {
        title: formTitle,
        body: formBody,
        tags,
      }

      const note = await api.createNote(input)
      setNotes([...notes, note])
      saveNoteId(note.id)
      setSelectedNote(null)
      setIsEditing(false)
      setFormTitle("")
      setFormBody("")
      setFormTags("")
    } catch (err) {
      setError(err instanceof Error ? err.message : "Failed to create note")
    } finally {
      setLoading(false)
    }
  }

  const handleUpdateNote = async () => {
    if (!selectedNote) return

    setLoading(true)
    setError(null)

    try {
      const tags = formTags
        .split(",")
        .map(t => t.trim())
        .filter(t => t)
      const input: UpdateNoteInput = {
        title: formTitle,
        body: formBody,
        tags,
        archived: selectedNote.archived,
      }

      const updated = await api.updateNote(selectedNote.id, input)
      setSelectedNote(updated)
      setNotes(notes.map(n => (n.id === updated.id ? updated : n)))
      setIsEditing(false)
    } catch (err) {
      setError(err instanceof Error ? err.message : "Failed to update note")
    } finally {
      setLoading(false)
    }
  }

  const handleDeleteNote = async () => {
    if (!selectedNote) return

    if (!confirm("Are you sure you want to delete this note?")) return

    setLoading(true)
    setError(null)

    try {
      await api.deleteNote(selectedNote.id)
      removeNoteId(selectedNote.id)
      setNotes(notes.filter(n => n.id !== selectedNote.id))
      setSelectedNote(null)
      setIsEditing(false)
      setFormTitle("")
      setFormBody("")
      setFormTags("")
    } catch (err) {
      setError(err instanceof Error ? err.message : "Failed to delete note")
    } finally {
      setLoading(false)
    }
  }

  const startEdit = () => {
    if (selectedNote) {
      setFormTitle(selectedNote.title)
      setFormBody(selectedNote.body)
      setFormTags(selectedNote.tags.join(", "))
      setIsEditing(true)
    }
  }

  const cancelEdit = () => {
    setIsEditing(false)
    setSelectedNote(null)
    setFormTitle("")
    setFormBody("")
    setFormTags("")
  }

  useEffect(() => {
    if (selectedNote && !isEditing) {
      setFormTitle(selectedNote.title)
      setFormBody(selectedNote.body)
      setFormTags(selectedNote.tags.join(", "))
    } else if (!selectedNote) {
      setFormTitle("")
      setFormBody("")
      setFormTags("")
      setIsEditing(false)
    }
  }, [selectedNote, isEditing])

  return (
    <div className="w-full p-6">
      <h1 className="text-4xl font-bold text-gray-900 dark:text-white mb-6">
        Note Taking App
      </h1>

      {error && (
        <div className="mb-4 p-4 bg-red-100 dark:bg-red-900 border border-red-400 text-red-700 dark:text-red-200 rounded">
          {error}
        </div>
      )}

      <div className="grid grid-cols-1 md:grid-cols-3 gap-6">
        <div className="md:col-span-2">
          <div className="bg-white dark:bg-gray-800 rounded-lg shadow p-6">
            <h2 className="text-2xl font-semibold text-gray-900 dark:text-white mb-4">
              {isEditing ? "Edit Note" : "Create New Note"}
            </h2>

            <form
              onSubmit={
                isEditing
                  ? e => {
                      e.preventDefault()
                      handleUpdateNote().catch(err => {
                        console.error("Update note error:", err)
                      })
                    }
                  : handleCreateNote
              }
            >
              <div className="mb-4">
                <label htmlFor="note-title" className="block text-sm font-medium text-gray-700 dark:text-gray-300 mb-2">
                  Title
                </label>
                <input
                  id="note-title"
                  type="text"
                  value={formTitle}
                  onChange={e => setFormTitle(e.target.value)}
                  required
                  className="w-full px-3 py-2 border border-gray-300 dark:border-gray-600 rounded-md shadow-sm focus:outline-none focus:ring-2 focus:ring-blue-500 dark:bg-gray-700 dark:text-white"
                />
              </div>

              <div className="mb-4">
                <label htmlFor="note-body" className="block text-sm font-medium text-gray-700 dark:text-gray-300 mb-2">
                  Body
                </label>
                <textarea
                  id="note-body"
                  value={formBody}
                  onChange={e => setFormBody(e.target.value)}
                  required
                  rows={8}
                  className="w-full px-3 py-2 border border-gray-300 dark:border-gray-600 rounded-md shadow-sm focus:outline-none focus:ring-2 focus:ring-blue-500 dark:bg-gray-700 dark:text-white"
                />
              </div>

              <div className="mb-4">
                <label htmlFor="note-tags" className="block text-sm font-medium text-gray-700 dark:text-gray-300 mb-2">
                  Tags (comma-separated)
                </label>
                <input
                  id="note-tags"
                  type="text"
                  value={formTags}
                  onChange={e => setFormTags(e.target.value)}
                  placeholder="tag1, tag2, tag3"
                  className="w-full px-3 py-2 border border-gray-300 dark:border-gray-600 rounded-md shadow-sm focus:outline-none focus:ring-2 focus:ring-blue-500 dark:bg-gray-700 dark:text-white"
                />
              </div>

              <div className="flex gap-2">
                {isEditing ? (
                  <>
                    <button
                      type="submit"
                      disabled={loading}
                      className="px-4 py-2 bg-blue-600 text-white rounded-md hover:bg-blue-700 disabled:opacity-50"
                    >
                      {loading ? "Saving..." : "Save"}
                    </button>
                    <button
                      type="button"
                      onClick={cancelEdit}
                      disabled={loading}
                      className="px-4 py-2 bg-gray-300 dark:bg-gray-600 text-gray-700 dark:text-gray-200 rounded-md hover:bg-gray-400 disabled:opacity-50"
                    >
                      Cancel
                    </button>
                    <button
                      type="button"
                      onClick={() => {
                        handleDeleteNote().catch(err => {
                          console.error("Delete note error:", err)
                        })
                      }}
                      disabled={loading}
                      className="px-4 py-2 bg-red-600 text-white rounded-md hover:bg-red-700 disabled:opacity-50 ml-auto"
                    >
                      Delete
                    </button>
                  </>
                ) : (
                  <button
                    type="submit"
                    disabled={loading}
                    className="px-4 py-2 bg-blue-600 text-white rounded-md hover:bg-blue-700 disabled:opacity-50"
                  >
                    {loading ? "Creating..." : "Create Note"}
                  </button>
                )}
              </div>
            </form>
          </div>
        </div>

        <div className="space-y-4">
          <div className="bg-white dark:bg-gray-800 rounded-lg shadow p-4">
            <h3 className="text-lg font-semibold text-gray-900 dark:text-white mb-3">
              Your Notes ({savedIds.length})
            </h3>
            <div className="space-y-2 max-h-64 overflow-y-auto">
              {savedIds.length === 0 ? (
                <p className="text-gray-500 dark:text-gray-400 text-sm">
                  No notes yet
                </p>
              ) : (
                savedIds.map(id => (
                  <button
                    key={id}
                    onClick={() => {
                      loadNote(id).catch(err => {
                        console.error("Load note error:", err)
                      })
                    }}
                    className="w-full text-left px-3 py-2 bg-gray-100 dark:bg-gray-700 hover:bg-gray-200 dark:hover:bg-gray-600 rounded text-sm text-gray-900 dark:text-white"
                  >
                    Note #{id}
                  </button>
                ))
              )}
            </div>
          </div>

          {selectedNote && !isEditing && (
            <div className="bg-white dark:bg-gray-800 rounded-lg shadow p-4">
              <div className="flex justify-between items-start mb-3">
                <h3 className="text-lg font-semibold text-gray-900 dark:text-white">
                  Note Details
                </h3>
                <button
                  onClick={startEdit}
                  className="text-sm text-blue-600 dark:text-blue-400 hover:underline"
                >
                  Edit
                </button>
              </div>
              <div className="space-y-2 text-sm">
                <p className="text-gray-600 dark:text-gray-300">
                  <span className="font-medium">ID:</span> {selectedNote.id}
                </p>
                <p className="text-gray-600 dark:text-gray-300">
                  <span className="font-medium">Title:</span>{" "}
                  {selectedNote.title}
                </p>
                <p className="text-gray-600 dark:text-gray-300">
                  <span className="font-medium">Body:</span> {selectedNote.body}
                </p>
                <p className="text-gray-600 dark:text-gray-300">
                  <span className="font-medium">Tags:</span>{" "}
                  {selectedNote.tags.join(", ") || "None"}
                </p>
                <p className="text-gray-600 dark:text-gray-300">
                  <span className="font-medium">Archived:</span>{" "}
                  {selectedNote.archived ? "Yes" : "No"}
                </p>
                <p className="text-gray-600 dark:text-gray-300">
                  <span className="font-medium">Updated:</span>{" "}
                  {new Date(selectedNote.updated_at).toLocaleString()}
                </p>
              </div>
            </div>
          )}
        </div>
      </div>
    </div>
  )
}

"""

In [ ]:
# Base test file - some of the app functionality is already tested

BASE_TEST_FILE = """
import { test, expect } from "@playwright/test"

test.describe("Note Taking App", () => {
  test.beforeEach(async ({ page }) => {
    await page.goto("/")
    await page.evaluate(() => localStorage.clear())
    await page.reload()
  })

  test("has title and create form visible", async ({ page }) => {
    await expect(
      page.getByRole("heading", { name: "Note Taking App" }),
    ).toBeVisible()
    await expect(
      page.getByRole("heading", { name: /Create New Note/ }),
    ).toBeVisible()
    await expect(
      page.getByRole("button", { name: "Create Note" }),
    ).toBeVisible()
  })

  test("can create a note and see it in the sidebar", async ({ page }) => {
    await page.getByLabel("Title").fill("My first note")
    await page.getByLabel("Body").fill("Some body text")
    await page.getByLabel(/Tags/).fill("demo, e2e")
    await page.getByRole("button", { name: "Create Note" }).click()

    await expect(page.getByRole("button", { name: /Note #\d+/ })).toBeVisible()
    await expect(page.getByText("Your Notes (1)")).toBeVisible()
  })

  test("can open a note and see its details", async ({ page }) => {
    await page.getByLabel("Title").fill("Detail test")
    await page.getByLabel("Body").fill("Content to verify")
    await page.getByLabel(/Tags/).fill("tag1")
    await page.getByRole("button", { name: "Create Note" }).click()

    await page.getByRole("button", { name: /Note #\d+/ }).click()

    await expect(
      page.getByRole("heading", { name: "Note Details" }),
    ).toBeVisible()
    await expect(page.getByText("Title: Detail test")).toBeVisible()
    await expect(page.getByText("Body: Content to verify")).toBeVisible()
  })
})

"""

**Only code changes**

In [ ]:
# First change diff (this was extracted from running the custom GitHub action and copying the generated diff from that)
FIRST_DIFF_CODE_ONLY_DIFF = """
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
+  const clearAllNotes = () => {
+    setSavedIds([])
+    setSelectedNote(null)
+    setFormTitle("")
+    setFormBody("")
+    setFormTags("")
+    setIsEditing(false)
+    localStorage.setItem("noteIds", JSON.stringify([]))
+  }
+
+      <p className="text-gray-500 dark:text-gray-400 text-sm mb-6">
+        Create and manage notes. Your notes are stored locally.
+      </p>
+
-            <h3 className="text-lg font-semibold text-gray-900 dark:text-white mb-3">
-              Your Notes ({savedIds.length})
-            </h3>
+            <div className="flex justify-between items-center mb-3">
+              <h3 className="text-lg font-semibold text-gray-900 dark:text-white">
+                Your Notes ({savedIds.length})
+              </h3>
+              {savedIds.length > 0 && (
+                <button
+                  type="button"
+                  onClick={clearAllNotes}
+                  className="text-sm text-red-600 dark:text-red-400 hover:underline"
+                >
+                  Clear all
+                </button>
+              )}
+            </div>
"""

In [ ]:
# Second change diff (this was extracted from running the custom GitHub action and copying the generated diff from that)
SECOND_DIFF_CODE_ONLY_DIFF = """
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
+                <p className="text-xs text-gray-500 dark:text-gray-400 mt-1">
+                  {formBody.length} / 5000
+                </p>
"""

In [ ]:
# Third change diff (this was extracted from running the custom GitHub action and copying the generated diff from that)
THIRD_DIFF_CODE_ONLY_DIFF = """
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
+  const copyNoteBody = () => {
+    if (selectedNote && navigator.clipboard) {
+      navigator.clipboard.writeText(selectedNote.body)
+    }
+  }
+
+                <button
+                  type="button"
+                  onClick={copyNoteBody}
+                  className="text-sm text-blue-600 dark:text-blue-400 hover:underline">
+                  Copy
+                </button>
"""

In [ ]:
# Fourth change diff (this was extracted from running the custom GitHub action and copying the generated diff from that)
FOURTH_DIFF_CODE_ONLY_DIFF = """
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
-import React from "react"
+import React, { useMemo } from "react"
+  const [tagFilter, setTagFilter] = useState("")
+  const [allNotes, setAllNotes] = useState<Note[]>([])
+  useEffect(() => {
+    api.getNotes().then(setAllNotes).catch(() => setAllNotes([]))
+  }, [savedIds])
+
+  const displayedIds = useMemo(() => {
+    if (!tagFilter.trim()) return savedIds
+    const lower = tagFilter.trim().toLowerCase()
+    return allNotes
+      .filter(n => savedIds.includes(n.id) && n.tags.some(t => t.toLowerCase().includes(lower)))
+      .map(n => n.id)
+  }, [savedIds, allNotes, tagFilter])
+
-              Your Notes ({savedIds.length})
+              Your Notes ({displayedIds.length})
+            <input
+              type="text"
+              placeholder="Filter by tag"
+              value={tagFilter}
+              onChange={e => setTagFilter(e.target.value)}
+              className="w-full px-2 py-1 text-sm border border-gray-300 dark:border-gray-600 rounded mb-2 dark:bg-gray-700 dark:text-white focus:outline-none focus:ring-1 focus:ring-blue-500"
+            />
-              {savedIds.length === 0 ? (
+              {displayedIds.length === 0 ? (
-                  No notes yet
+                  {savedIds.length === 0
+                    ? "No notes yet"
+                    : "No notes match the filter"}
-                savedIds.map(id => (
+                displayedIds.map(id => (
"""

Now diffs for code and tests

In [ ]:
# First change diff where both app and test code were changed (this was extracted from running the custom GitHub action and copying the generated diff from that)
FIRST_DIFF_CODE_AND_TESTS_DIFF = """
--- a/app/e2e/Home.spec.ts
+++ b/app/e2e/Home.spec.ts
+    await expect(
+      page.getByText("Create and manage notes. Your notes are stored locally."),
+    ).toBeVisible()
+
+  test("can clear all notes from sidebar", async ({ page }) => {
+    await page.getByLabel("Title").fill("To clear")
+    await page.getByLabel("Body").fill("Body")
+    await page.getByLabel(/Tags/).fill("x")
+    await page.getByRole("button", { name: "Create Note" }).click()
+    await expect(page.getByText("Your Notes (1)")).toBeVisible()
+    await page.getByRole("button", { name: "Clear all" }).click()
+    await expect(page.getByText("No notes yet")).toBeVisible()
+    await expect(page.getByText("Your Notes (0)")).toBeVisible()
+  })
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
+  const clearAllNotes = () => {
+    setSavedIds([])
+    setSelectedNote(null)
+    setFormTitle("")
+    setFormBody("")
+    setFormTags("")
+    setIsEditing(false)
+    localStorage.setItem("noteIds", JSON.stringify([]))
+  }
+
+      <p className="text-gray-500 dark:text-gray-400 text-sm mb-6">
+        Create and manage notes. Your notes are stored locally.
+      </p>
+
-            <h3 className="text-lg font-semibold text-gray-900 dark:text-white mb-3">
-              Your Notes ({savedIds.length})
-            </h3>
+            <div className="flex justify-between items-center mb-3">
+              <h3 className="text-lg font-semibold text-gray-900 dark:text-white">
+                Your Notes ({savedIds.length})
+              </h3>
+              {savedIds.length > 0 && (
+                <button
+                  type="button"
+                  onClick={clearAllNotes}
+                  className="text-sm text-red-600 dark:text-red-400 hover:underline"
+                >
+                  Clear all
+                </button>
+              )}
+            </div>
"""

In [ ]:
# Second change diff where both app and test code were changed (this was extracted from running the custom GitHub action and copying the generated diff from that)
SECOND_DIFF_CODE_AND_TESTS_DIFF = """
--- a/app/e2e/Home.spec.ts
+++ b/app/e2e/Home.spec.ts
+
+  test("shows body character count and updates when typing", async ({ page }) => {
+    await expect(page.getByText("0 / 5000")).toBeVisible()
+    await page.getByLabel("Body").fill("Hello")
+    await expect(page.getByText("5 / 5000")).toBeVisible()
+    await page.getByLabel("Body").fill("Hello world")
+    await expect(page.getByText("11 / 5000")).toBeVisible()
+  })
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
+                <p className="text-xs text-gray-500 dark:text-gray-400 mt-1">
+                  {formBody.length} / 5000
+                </p>
"""

In [ ]:
# Third change diff where both app and test code were changed (this was extracted from running the custom GitHub action and copying the generated diff from that)

THIRD_DIFF_CODE_AND_TESTS_DIFF = """
--- a/app/e2e/Home.spec.ts
+++ b/app/e2e/Home.spec.ts
+
+  test("Copy button copies note body to clipboard", async ({ page, context }) => {
+    await context.grantPermissions(["clipboard-read", "clipboard-write"])
+
+    const bodyText = "Text to copy from note"
+    await page.getByLabel("Title").fill("Copy test")
+    await page.getByLabel("Body").fill(bodyText)
+    await page.getByLabel(/Tags/).fill("tag1")
+    await page.getByRole("button", { name: "Create Note" }).click()
+
+    await page.getByRole("button", { name: /Note #\d+/ }).click()
+    await expect(page.getByRole("heading", { name: "Note Details" })).toBeVisible()
+
+    await page.getByRole("button", { name: "Copy" }).click()
+
+    const clipboardText = await page.evaluate(() => navigator.clipboard.readText())
+    expect(clipboardText).toBe(bodyText)
+  })
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
+  const copyNoteBody = () => {
+    if (selectedNote && navigator.clipboard) {
+      navigator.clipboard.writeText(selectedNote.body)
+    }
+  }
+
+                <button
+                  type="button"
+                  onClick={copyNoteBody}
+                  className="text-sm text-blue-600 dark:text-blue-400 hover:underline">
+                  Copy
+                </button>
"""

In [ ]:
# Fourth change diff where both app and test code were changed (this was extracted from running the custom GitHub action and copying the generated diff from that)

FOURTH_DIFF_CODE_AND_TESTS_DIFF = """
--- a/app/e2e/Home.spec.ts
+++ b/app/e2e/Home.spec.ts
+
+  test("filter by tag shows only matching notes", async ({ page }) => {
+    await page.getByLabel("Title").fill("Work note")
+    await page.getByLabel("Body").fill("Body one")
+    await page.getByLabel(/Tags/).fill("work, urgent")
+    await page.getByRole("button", { name: "Create Note" }).click()
+
+    await page.getByLabel("Title").fill("Personal note")
+    await page.getByLabel("Body").fill("Body two")
+    await page.getByLabel(/Tags/).fill("personal")
+    await page.getByRole("button", { name: "Create Note" }).click()
+
+    await expect(page.getByText("Your Notes (2)")).toBeVisible()
+
+    await page.getByPlaceholder("Filter by tag").fill("work")
+    await expect(page.getByText("Your Notes (1)")).toBeVisible()
+    await expect(page.getByRole("button", { name: "Note #1" })).toBeVisible()
+    await expect(page.getByRole("button", { name: "Note #2" })).not.toBeVisible()
+
+    await page.getByPlaceholder("Filter by tag").fill("nonexistent")
+    await expect(page.getByText("No notes match the filter")).toBeVisible()
+    await expect(page.getByText("Your Notes (0)")).toBeVisible()
+
+    await page.getByPlaceholder("Filter by tag").fill("")
+    await expect(page.getByText("Your Notes (2)")).toBeVisible()
+  })
--- a/app/src/pages/Home.tsx
+++ b/app/src/pages/Home.tsx
-import React from "react"
+import React, { useMemo } from "react"
+  const [tagFilter, setTagFilter] = useState("")
+  const [allNotes, setAllNotes] = useState<Note[]>([])
+  useEffect(() => {
+    api.getNotes().then(setAllNotes).catch(() => setAllNotes([]))
+  }, [savedIds])
+
+  const displayedIds = useMemo(() => {
+    if (!tagFilter.trim()) return savedIds
+    const lower = tagFilter.trim().toLowerCase()
+    return allNotes
+      .filter(n => savedIds.includes(n.id) && n.tags.some(t => t.toLowerCase().includes(lower)))
+      .map(n => n.id)
+  }, [savedIds, allNotes, tagFilter])
+
-              Your Notes ({savedIds.length})
+              Your Notes ({displayedIds.length})
+            <input
+              type="text"
+              placeholder="Filter by tag"
+              value={tagFilter}
+              onChange={e => setTagFilter(e.target.value)}
+              className="w-full px-2 py-1 text-sm border border-gray-300 dark:border-gray-600 rounded mb-2 dark:bg-gray-700 dark:text-white focus:outline-none focus:ring-1 focus:ring-blue-500"
+            />
-              {savedIds.length === 0 ? (
+              {displayedIds.length === 0 ? (
-                  No notes yet
+                  {savedIds.length === 0
+                    ? "No notes yet"
+                    : "No notes match the filter"}
-                savedIds.map(id => (
+                displayedIds.map(id => (
"""

In [ ]:
# Open Source Models Tested

# Qwen/Qwen2.5-Coder-14B-Instruct 14.7B (from Alibaba)
# mistralai/Mistral-Nemo-Instruct-2407  12.2B (from Mistral)
# deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct 16B (from DeepSeek)
# google/gemma-3-12b-it  12B (from Google)

# Qwen/Qwen2.5-Coder-32B-Instruct  32B (from Alibaba)
# mistralai/Mistral-Small-24B-Instruct-2501 24B (from Mistral)
# google/gemma-3-27b-it 27B (from Google)


# This was updated depending on the desired model for testing
model_id = "google/gemma-3-27b-it"

# Quantisation was required to ensure the models would fit on the A100 (particularly for the larger ones)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# HuggingFace Tokeniser defined
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Loading in the model from Hugging Face using the quantisation config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

# Defining a Hugging Facwe text generation pipeline with the defined model and tokeniser
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
# Defining variables here for easy use
PROMPT_TYPES = ["zero-shot", "few-shot", "instruction", "chain-of-thought"]
MODEL_NAME = "google-27B"
SCENARIOS = ["diff-only", "diff-and-source-code", "diff-and-source-code-and-tests"]
CURRENT_DIFF_IDENTIFIER = "FOURTH_DIFF_CODE_ONLY_DIFF"
CHANGES = [
    FIRST_DIFF_CODE_ONLY_DIFF,
    SECOND_DIFF_CODE_ONLY_DIFF,
    THIRD_DIFF_CODE_ONLY_DIFF,
    FOURTH_DIFF_CODE_ONLY_DIFF,
    FIRST_DIFF_CODE_AND_TESTS_DIFF,
    SECOND_DIFF_CODE_AND_TESTS_DIFF,
    THIRD_DIFF_CODE_AND_TESTS_DIFF,
    FOURTH_DIFF_CODE_AND_TESTS_DIFF
    ]

In [ ]:
def log_experiment_result(data, filename="benchmark_results.json"):
    """
    This function writes the passed data to a JSON file with the filename provided
    """

    # First make sure that the directory exists
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    # Empty list for the data
    logs = []

    # If the file already exists, read the data and append to the logs variable
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            try:
                logs = json.load(f)
                if not isinstance(logs, list):
                    logs = [logs]
            except json.JSONDecodeError:
                logs = []

    # Then add the new data
    logs.append(data)

    # Write the data to the file
    with open(filename, 'w') as f:
        json.dump(logs, f, indent=4)

In [ ]:
def get_variable_name(var):
    """
    This function was used to get the actual defined variable name (for auditing of what prompt type was used for each run)
    """

    # Loop over the globals until match found for passed variable name and it is of variable type
    for name, value in globals().items():
        if value is var:
            return name

In [ ]:
# Outer loop to go over all changes defined
for change in CHANGES:
  # For each change, loop over all prompt types
  for prompt_type in PROMPT_TYPES:

    # For each change and prompt type, loop over each scenario
    for scenario in SCENARIOS:

      # Initialise to None for these as they are not used for diff only tests
      base_code_file = None
      base_test_file = None

      # the base code file is provided as context for all apart from diff-only
      if scenario != "diff-only":
        base_code_file = BASE_CODE_FILE

      # the base test file is provided as context if type is diff, source code, and tests
      if scenario == "diff-and-source-code-and-tests":
        base_test_file = BASE_TEST_FILE

      # Get the current iteration prompt, injecting all relevant context into the prompt
      prompt = generate_refined_prompt(prompt_type, change, base_code_file, base_test_file)

      # Call the tokeniser to get the input token count for the prompt (pt returns as Pytorch tensors)
      input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
      input_token_count = input_ids.shape[1]
      model_name = MODEL_NAME

      # Capture the start time of inferencing so duration can be calculated after generation
      start_time = time.perf_counter()

      # To let user know of current step (as inferencing can take a long time depending on model)
      print(f"Starting inference for prompt type {prompt_type} for scenario {scenario} and model {model_id}")


      # Hugging Face pipeline called and result stored in variable
      # Note not much hyperparameter tuning in place (more timing could warrant an investigation there)
      # Max tokens of 1024 is required for more complex prompts such as chain-of-thought which
      # output a lot of reasoning text output before the actual test generation output
      sequences = pipe(
          prompt,
          max_new_tokens=1024,
          do_sample=False,
          return_full_text=False,
      )

      # Get inferencing end time so that duration can be calculated
      end_time = time.perf_counter()

      # The raw output extracted from the model response
      generated_text = sequences[0]['generated_text']

      # Getting output tokens count for another data point (to see differences across model)
      output_tokens = tokenizer(generated_text, return_tensors="pt")["input_ids"]
      output_token_count = output_tokens.shape[1]

      # Calculate the inference time
      inference_time = end_time - start_time

      # Calculate the tokens generated per second (useful addition to total inference time if avaialble)
      tokens_per_sec = output_token_count / inference_time if inference_time > 0 else 0

      try:
          # Try get the test generated code from the raw output
          generated_code = generated_text.split("### OUTPUT ONLY THE TYPESCRIPT CODE:")[1].strip()
      except IndexError:
          # If no match there, just stick to the full output
          generated_code = generated_text


      # Timestamping the file name ensures uniqueness
      timestamp = datetime.now().isoformat()

      # Define result dict with as much info as possible for reporting afterwards
      result = {
              "timestamp": timestamp,
              "model_id": model_id,
              "prompt_type": prompt_type,
              "scenario": scenario,
              "metrics": {
                  "inference_time_s": round(inference_time, 3),
                  "tokens_per_sec": round(tokens_per_sec, 2),
                  "input_token_count": input_token_count,
                  "output_token_count": output_token_count,
                  "total_token_count": input_token_count + output_token_count
              },
              "content": {
                  "prompt": prompt,
                  "raw_response": generated_text,
                  "extracted_code": generated_code
              },
              "hyperparameters": {
                  "max_new_tokens": 1024,
                  "do_sample": False,
                  "return_full_text": False,
              }
          }

      # Call helper function to save the results to disk
      log_experiment_result(result, f"results/refined_{model_name}_{prompt_type}_{scenario}_{timestamp}.json")

      # For debugging (to let me know that inferencing actually worked)
      print(f"Done! Inference took {inference_time:.2f}s for prompt type {prompt_type} for scenario {scenario} and model {model_id}")


  # Once all are done for a change type wait a few seconds (as there can be a lag writing the files to disk)
  time.sleep(5)

  # Timestamp generated for the zip file (again for uniquness)
  current_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

  # Output file name for the zipped directory
  zip_filename_base = f"benchmark_results_{MODEL_NAME}_{get_variable_name(change)}_{current_timestamp}"

  # Zipped archive of the results created
  zip_filepath = shutil.make_archive(zip_filename_base, 'zip', 'results')

  # Zipped folder downloaded to local machine
  files.download(zip_filepath)


  # Cleanup, before trying to clear, ensure that it exists for next run
  if not os.path.exists('results'):
    os.makedirs('results')

  # Remove everything from the results directory to ensure nexxt run has empty directory there
  for item in os.listdir('results'):
    item_path = os.path.join('results', item)
    if os.path.isfile(item_path) or os.path.islink(item_path):
        os.unlink(item_path)
    elif os.path.isdir(item_path):
        shutil.rmtree(item_path)

  time.sleep(5)

In [ ]:
import gc

# Delete model and tokeniser objects to free up memory across disk, GPU, RAM for next model run
del model
del tokenizer

# Garbage collection can speed this process up
gc.collect()

# Also clearing pytorch cache
torch.cuda.empty_cache()